# ID02 - mlops_pipeline_smoke_test

Smoke test end-to-end del pipeline de MLOps del repo: dataset sintético (sin ningún dato sensible ni real) → `dvc add`/`dvc push` → split/entrenamiento reusando código de `src/julieta` → `log_run` a MLflow sobre Azure ML → artefactos de resultado. Ver `README.md` de este experimento para el detalle completo.

In [1]:
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from sklearn.datasets import make_classification

from sklearn.metrics import accuracy_score

import julieta
from julieta.data.split_data import DataSplitter
from julieta.models.classifiers import BalancedXGBClassifier
from julieta.models.metrics import ClassificationMetrics, julieta_score
from julieta.tracking.mlflow_config import log_run
from julieta.utils.generate_experiment_report import build_report

REPO_ROOT = Path(julieta.__file__).resolve().parents[2]
EXPERIMENT_DIR = Path.cwd().resolve().parent
EXPERIMENT_ID, EXPERIMENT_NAME = EXPERIMENT_DIR.name.split("-", 1)

# El nombre del config tambien se usa como run_name en MLflow (ver celda 5)
# -- asi el run se identifica por que variante es, no por un nombre random.
CONFIG_NAME = "baseline"
with open(EXPERIMENT_DIR / "configs" / f"{CONFIG_NAME}.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)

REPO_ROOT, EXPERIMENT_DIR, EXPERIMENT_ID, EXPERIMENT_NAME

(PosixPath('/home/valentin/projects/julieta-experimentation'),
 PosixPath('/home/valentin/projects/julieta-experimentation/experiments/ID02-mlops_pipeline_smoke_test'),
 'ID02',
 'mlops_pipeline_smoke_test')

## 1. Generar dataset sintético y subirlo a DVC

In [2]:
ds_cfg = config["dataset"]
X, y = make_classification(
    n_samples=ds_cfg["n_samples"],
    n_features=ds_cfg["n_features"],
    n_informative=ds_cfg["n_informative"],
    n_redundant=ds_cfg["n_redundant"],
    weights=ds_cfg["class_weights"],
    random_state=ds_cfg["random_state"],
)
feature_names = [f"feature_{i}" for i in range(X.shape[1])]
df = pd.DataFrame(X, columns=feature_names)
df["target"] = y

data_path = REPO_ROOT / "data" / "raw" / "synthetic_smoke_test.csv"
data_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(data_path, index=False)

print(f"dataset: {df.shape}, balance clase positiva: {df['target'].mean():.3f}")

dataset: (2000, 13), balance clase positiva: 0.301


In [3]:
def run(cmd):
    result = subprocess.run(cmd, cwd=REPO_ROOT, capture_output=True, text=True)
    print(" ".join(cmd), "->", result.returncode)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        result.check_returncode()
    return result

rel_data_path = str(data_path.relative_to(REPO_ROOT))
run([sys.executable, "-m", "dvc", "add", rel_data_path])
run([sys.executable, "-m", "dvc", "push"])

/home/valentin/projects/julieta-experimentation/.venv/bin/python -m dvc add data/raw/synthetic_smoke_test.csv -> 0

To track the changes with git, run:

	git add data/raw/synthetic_smoke_test.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true



/home/valentin/projects/julieta-experimentation/.venv/bin/python -m dvc push -> 0
Everything is up to date.



CompletedProcess(args=['/home/valentin/projects/julieta-experimentation/.venv/bin/python', '-m', 'dvc', 'push'], returncode=0, stdout='Everything is up to date.\n', stderr='')

## 2. Split train/test

In [4]:
X_df = df[feature_names]
y_s = df["target"]

splitter = DataSplitter()
X_train, X_test, y_train, y_test = splitter.split(
    X_df,
    y_s,
    test_size=config["split"]["test_size"],
    stratify=config["split"]["stratify"],
)
X_train.shape, X_test.shape

((1600, 12), (400, 12))

## 3. Entrenar (`BalancedXGBClassifier`)

In [5]:
model_cfg = config["model"]
model = BalancedXGBClassifier(
    n_estimators=model_cfg["n_estimators"],
    max_depth=model_cfg["max_depth"],
    learning_rate=model_cfg["learning_rate"],
    random_state=model_cfg["random_state"],
    eval_metric=model_cfg["eval_metric"],
)
model.fit(X_train, y_train)

,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'
,feature_types,None


## 4. Evaluar (train y test)

Todas las métricas de `ClassificationMetrics` más `accuracy` y el `julieta_score` propio del repo, calculadas para train y para test por separado -- comparar ambas es lo que deja ver overfitting, no solo el número de test aislado.

In [6]:
def compute_all_metrics(y_true, y_pred, y_proba):
    cm = ClassificationMetrics(y_true=y_true, y_pred=y_pred, y_proba=y_proba)
    values = cm.get_metrics()
    values["accuracy"] = round(accuracy_score(y_true, y_pred), 2)
    values["julieta_score"] = round(julieta_score(y_true, y_pred), 2)
    return values, cm

y_train_pred = model.predict(X_train)
y_train_proba = model.predict_proba(X_train)[:, 1]
y_test_pred = model.predict(X_test)
y_test_proba = model.predict_proba(X_test)[:, 1]

train_values, cm_train = compute_all_metrics(y_train, y_train_pred, y_train_proba)
test_values, cm_test = compute_all_metrics(y_test, y_test_pred, y_test_proba)

metrics = {f"train_{k}": float(v) for k, v in train_values.items()}
metrics.update({f"test_{k}": float(v) for k, v in test_values.items()})
print(metrics)

{'train_roc_auc': 1.0, 'train_f1_macro': 0.98, 'train_sensitivity': 1.0, 'train_specificity': 0.98, 'train_ppv': 0.95, 'train_npv': 1.0, 'train_accuracy': 0.98, 'train_julieta_score': 0.99, 'test_roc_auc': 0.94, 'test_f1_macro': 0.85, 'test_sensitivity': 0.85, 'test_specificity': 0.88, 'test_ppv': 0.75, 'test_npv': 0.93, 'test_accuracy': 0.87, 'test_julieta_score': 0.86}


In [7]:
results_dir = EXPERIMENT_DIR / "results"
results_dir.mkdir(parents=True, exist_ok=True)

artifact_paths = {}
for split_name, cm_obj in [("train", cm_train), ("test", cm_test)]:
    fig_cm = cm_obj.plot_confusion_matrix(class_names=["Clase 0", "Clase 1"])
    cm_path = results_dir / f"confusion_matrix_{split_name}.png"
    fig_cm.savefig(cm_path, dpi=150, bbox_inches="tight")
    plt.close(fig_cm)
    artifact_paths[f"confusion_matrix_{split_name}"] = cm_path

    fig_report = cm_obj.plot_classification_report()
    report_path = results_dir / f"classification_report_{split_name}.png"
    fig_report.savefig(report_path, dpi=150, bbox_inches="tight")
    plt.close(fig_report)
    artifact_paths[f"classification_report_{split_name}"] = report_path

print([p.name for p in artifact_paths.values()])

['confusion_matrix_train.png', 'classification_report_train.png', 'confusion_matrix_test.png', 'classification_report_test.png']


## 5. Loguear el run en MLflow (Azure ML)

In [8]:
flat_config = {
    **{f"dataset.{k}": v for k, v in config["dataset"].items()},
    **{f"split.{k}": v for k, v in config["split"].items()},
    **{f"model.{k}": v for k, v in config["model"].items()},
}

run_id = log_run(
    experiment_id=EXPERIMENT_ID,
    experiment_name=EXPERIMENT_NAME,
    author="dgrajales",
    config=flat_config,
    metrics=metrics,
    artifacts=[str(p) for p in artifact_paths.values()],
    tags={"purpose": "mlops_platform_smoke_test", "data_sensitivity": "level_0_synthetic"},
    run_name=CONFIG_NAME,
)
print("MLflow run_id:", run_id, "| run_name:", CONFIG_NAME)

2026/09/08 09:29:16 INFO mlflow.tracking.fluent: Experiment with name 'julieta/ID02-mlops_pipeline_smoke_test' does not exist. Creating a new experiment.


🏃 View run baseline at: https://eastus.api.azureml.ms/mlflow/v2.0/subscriptions/346d9d5d-907a-489f-922b-c221e9a44b20/resourceGroups/ml-ops/providers/Microsoft.MachineLearningServices/workspaces/ml-salva-dev/#/experiments/1c686091-7b53-4582-b288-41391d463c05/runs/6d95d186-77f5-436d-adf6-140d0ccbc6a2
🧪 View experiment at: https://eastus.api.azureml.ms/mlflow/v2.0/subscriptions/346d9d5d-907a-489f-922b-c221e9a44b20/resourceGroups/ml-ops/providers/Microsoft.MachineLearningServices/workspaces/ml-salva-dev/#/experiments/1c686091-7b53-4582-b288-41391d463c05


MLflow run_id: 6d95d186-77f5-436d-adf6-140d0ccbc6a2 | run_name: baseline


In [9]:
summary = {
    "run_id": run_id,
    "run_name": CONFIG_NAME,
    "experiment_full_name": f"julieta/{EXPERIMENT_ID}-{EXPERIMENT_NAME}",
    "metrics": metrics,
    "artifacts": [p.name for p in artifact_paths.values()],
}
with open(results_dir / "run_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print(json.dumps(summary, indent=2, ensure_ascii=False))

{
  "run_id": "6d95d186-77f5-436d-adf6-140d0ccbc6a2",
  "run_name": "baseline",
  "experiment_full_name": "julieta/ID02-mlops_pipeline_smoke_test",
  "metrics": {
    "train_roc_auc": 1.0,
    "train_f1_macro": 0.98,
    "train_sensitivity": 1.0,
    "train_specificity": 0.98,
    "train_ppv": 0.95,
    "train_npv": 1.0,
    "train_accuracy": 0.98,
    "train_julieta_score": 0.99,
    "test_roc_auc": 0.94,
    "test_f1_macro": 0.85,
    "test_sensitivity": 0.85,
    "test_specificity": 0.88,
    "test_ppv": 0.75,
    "test_npv": 0.93,
    "test_accuracy": 0.87,
    "test_julieta_score": 0.86
  },
  "artifacts": [
    "confusion_matrix_train.png",
    "classification_report_train.png",
    "confusion_matrix_test.png",
    "classification_report_test.png"
  ]
}


## 6. Generar el reporte (`reports/report.html`)

In [10]:
report_path = build_report(EXPERIMENT_DIR)
print(f"reporte: {report_path}")

reporte: /home/valentin/projects/julieta-experimentation/experiments/ID02-mlops_pipeline_smoke_test/reports/report.html
